# EvoFlow use case: NK-cell signalling knock-outs

This notebook is an end-to-end vignette for **EvoFlow**. It was written to run
inside the R + Python Jupyter container published with the project:

```bash
docker run --rm -p 8888:8888 ghcr.io/vergaju/evoflow:latest
```

That image already contains Python, R, **NicheNet** (with its Bioconductor
dependencies) and EvoFlow, so no local installation is required. Open
`vignettes/use_case.ipynb` inside JupyterLab and run all cells.

The workflow mirrors the reference Snakemake pipeline:

1. **R / NicheNet** — infer a weighted ligand-to-target signalling network from
   a short table of receptors and target transcription factors.
2. **Expression data** — load the `kang_2018` PBMC dataset with `pertpy`, keep
   the NK cells and build the samples x genes matrix used to weight capacities.
3. **EvoFlow** — prepare the flow network and run the genetic algorithm.
4. **Interpretation** — compare the balanced, flow, cost and redundancy
   solutions.

> **First run.** NicheNet downloads its reference networks from Zenodo (about
> 1 GB) and `pertpy` downloads the `kang_2018` dataset (about 38 MB). Both are
> cached for later runs.
>
> **Figures.** Plotting is fixed at 96 DPI with small figure sizes so the
> committed notebook stays lightweight in git.

## 0. Setup

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

import EvoFlow

# Keep every figure small and low-resolution (the notebook is committed to git).
plt.rcParams["figure.dpi"] = 96
plt.rcParams["savefig.dpi"] = 96
plt.rcParams["figure.figsize"] = (6, 4)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)

print("EvoFlow:", EvoFlow.__version__)
print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())
print("Rscript:", shutil.which("Rscript"))

## 1. Infer the signalling network with NicheNet (R)

NicheNet needs two short lists:

- **receptors** expressed by the receiver cells (here NK cells),
- **targets**, i.e. transcription factors whose downstream activity we want to
  modulate.

The helper script `scripts/get_nichenet_network.R` derives the candidate
ligands from NicheNet's ligand-receptor network, keeps the signalling paths that
reach the requested targets and writes a weighted network plus a node
annotation table. This is the same script used by the Snakemake pipeline.

In [ ]:
WORKDIR = Path.cwd()
INPUT_DIR = WORKDIR / "inputs"
OUTPUT_DIR = WORKDIR / "outputs"
NICHENET_DIR = OUTPUT_DIR / "nichenet"
for directory in (INPUT_DIR, OUTPUT_DIR, NICHENET_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Receptors present in NicheNet's ligand-receptor network and relevant to NK
# cells, plus NK-lineage transcription factors used as targets.
RECEPTORS = ["IL2RB", "IL2RG", "IL15RA", "KLRK1", "IFNGR1", "IL12RB1"]
TARGETS = ["TBX21", "EOMES", "STAT1", "IRF1", "STAT4", "PRDM1"]

SAMPLE = "nk_signalling"
inputs_path = INPUT_DIR / f"{SAMPLE}.csv"
pd.DataFrame({"receptors": RECEPTORS, "targets": TARGETS}).to_csv(
    inputs_path, index=False
)
inputs_path

Run the R step through `Rscript`. On the first run NicheNet downloads its
reference networks; later runs reuse the cached `db/matrices.rda` file.

> Any R plots created inside the container should use
> `options(repr.plot.res = 96, repr.plot.width = 6, repr.plot.height = 4)` to
> stay low-resolution. This notebook only consumes the TSV outputs, so no R
> figures are embedded here.

In [ ]:
script_candidates = [
    WORKDIR / "scripts" / "get_nichenet_network.R",
    WORKDIR / "vignettes" / "scripts" / "get_nichenet_network.R",
    Path("/home/jovyan/work/vignettes/scripts/get_nichenet_network.R"),
]
r_script = next((path for path in script_candidates if path.exists()), None)
if r_script is None:
    raise FileNotFoundError(
        "get_nichenet_network.R not found: run this notebook from the "
        "vignettes directory of the EvoFlow container image."
    )

command = [
    "Rscript",
    str(r_script),
    "--input",
    str(inputs_path),
    "--output_dir",
    str(NICHENET_DIR),
    "--top_n_regulators",
    "4",
    "--consider_receptors",
    "false",
]
print("$", " ".join(command))
process = subprocess.run(command, capture_output=True, text=True)
print(process.stdout)
if process.returncode != 0:
    print(process.stderr, file=sys.stderr)
process.check_returncode()

In [ ]:
network_path = NICHENET_DIR / f"{SAMPLE}_weighted_signaling_network.tsv"
annotation_path = NICHENET_DIR / f"{SAMPLE}_annotation_table.tsv"

network_df = pd.read_csv(network_path, sep="\t")
annotation_df = pd.read_csv(annotation_path, sep="\t")

print(f"network: {network_df.shape[0]} edges, {network_df.shape[1]} columns")
print(f"annotation: {annotation_df['gene'].nunique()} genes")
display(network_df.head())
display(annotation_df["annotation"].value_counts())

## 2. Expression data: NK cells from `kang_2018`

EvoFlow converts expression into one **percolation** value per gene, which
scales edge capacities. We use `pertpy` to download the Kang et al. (2018)
PBMC dataset, keep the `NK cells` and restrict the matrix to the genes present
in the NicheNet annotation. Subsetting before densifying keeps the matrix small
(the full dataset is about 25k cells x 16k genes).

In [ ]:
import pertpy as pt

adata = pt.dt.kang_2018()
print(adata)
display(adata.obs["cell_type"].value_counts())

In [ ]:
annotation_genes = set(annotation_df["gene"].astype(str)) & set(adata.var_names)
genes = sorted(annotation_genes)
print(
    f"{len(genes)} of {annotation_df['gene'].nunique()} annotated genes are "
    "measured in kang_2018"
)

nk = adata[adata.obs["cell_type"].astype(str) == "NK cells", genes].copy()
nk.obs["condition"] = nk.obs["label"].astype(str)
print("NK cells:", nk.n_obs, "| genes:", nk.n_vars)
display(nk.obs["condition"].value_counts())

values = nk.X.toarray() if hasattr(nk.X, "toarray") else np.asarray(nk.X)
expression_df = pd.DataFrame(values, index=nk.obs_names, columns=nk.var_names)
expression_path = INPUT_DIR / "kang_2018_nk_expression.csv"
expression_df.to_csv(expression_path)
print("saved", expression_path, expression_df.shape)

In [ ]:
markers = [gene for gene in ["NKG7", "GNLY", "PRF1", "GZMB"] if gene in nk.var_names]
if markers:
    sc.pl.violin(nk, markers, groupby="condition", show=False, stripplot=False)
    plt.gcf().set_size_inches(6, 3)
    plt.show()

## 3. Run EvoFlow on the inferred network

The low-level API mirrors the reference pipeline: load the network,
annotation and expression matrix, compute percolation, build the graph, update
capacities and prepare the `SuperSource`/`SuperSink` topology. `lock_targets=1`
protects the original receptor nodes from removal, matching the Snakemake
configuration.

In [ ]:
from EvoFlow.biology import BioNetworkPreparer

preparer = BioNetworkPreparer(lock_targets=1)
preparer.load_data(
    network_path=str(network_path),
    annotation_path=str(annotation_path),
    expression_path=str(expression_path),
)
preparer.compute_percolation(stat="nonzero_ratio")
preparer.build_base_network()
preparer.update_edge_capacities(method="multiplicative")

# Keep only the receptors/targets that survived NicheNet's network filtering.
receptors = [gene for gene in RECEPTORS if gene in preparer.G]
targets = [gene for gene in TARGETS if gene in preparer.G]
print("sources:", receptors)
print("sinks:", targets)
if not receptors or not targets:
    raise RuntimeError(
        "The NicheNet network contains none of the selected receptors/targets. "
        "Inspect the annotation table and adjust RECEPTORS/TARGETS."
    )

sources, sinks = preparer.prepare_bioflow_topology(
    receptors=receptors, target_genes=targets
)
network = preparer.G
print(f"graph: {network.number_of_nodes()} nodes, {network.number_of_edges()} edges")

The genetic algorithm removes a small budget of nodes (5% of the network by
default here) while minimising flow, removal cost and SCC redundancy. Parameters
are set explicitly so the run is short and reproducible; omit
`population_size`/`early_stop_gen` to let the bundled GAM choose them.

EvoFlow writes its artifacts (`ga_run_*.jsonl`) to the current working
directory, so we run it inside `outputs/evoflow`.

In [ ]:
from EvoFlow.core import GAOptimizer

budget = max(2, int(round(0.05 * network.number_of_nodes())))
print("removal budget:", budget)

optimizer = GAOptimizer(
    network,
    sources,
    sinks,
    max_remove=budget,
    delta_penalty=False,
    repair=True,
    dynamic=True,
    use_multiprocessing=False,  # deterministic and safe inside Jupyter
)

evoflow_dir = OUTPUT_DIR / "evoflow"
evoflow_dir.mkdir(parents=True, exist_ok=True)
previous_cwd = Path.cwd()
os.chdir(evoflow_dir)
try:
    results, logs, population, norm = optimizer.run(
        generations=200,
        population_size=50,
        early_stop_gen=20,
        guess_parameters=False,
        verbose=True,
    )
finally:
    os.chdir(previous_cwd)
print("optimization complete")

## 4. Inspect the results

In [ ]:
rows = []
for solution, payload in results.items():
    row = {
        "solution": solution,
        "num_removed": payload["num_removed"],
        **{name: round(value, 3) for name, value in payload["fitness"].items()},
        "removed_nodes": ", ".join(payload["removed_nodes"]),
    }
    rows.append(row)
summary = pd.DataFrame(rows).set_index("solution")
display(summary)

In [ ]:
logs_df = pd.DataFrame(logs).T.reset_index(names="gen").sort_values("gen")
logs_df["gen"] = logs_df["gen"].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, metric in zip(axes, ["Flow", "Redundancy"]):
    if metric in logs_df:
        ax.plot(logs_df["gen"], logs_df[metric], linewidth=1.2)
        ax.set_title(metric)
        ax.set_xlabel("generation")
        ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
from EvoFlow.visualization import plot_graph

final_graph = optimizer.SG.copy()
final_graph.remove_nodes_from(results["Best_balanced"]["removed_nodes"])
fig, ax = plot_graph(final_graph, filename_prefix=None, layout="spring", figsize=(6, 4))
plt.show()

### How to read the table

- **Flow** is the remaining maximum flow after removing the selected nodes;
  lower is better.
- **Node** is the number of removed nodes (the cost of the intervention).
- **Redundancy** counts strongly connected components that remain after the
  removal, i.e. residual redundant signalling.
- `Best_balanced` is the Pareto solution closest to the origin after
  per-objective normalisation, `Best_Flow` minimises flow, `Best_Cost`
  minimises the number of removed nodes and `Best_Redundancy` minimises
  redundancy.

Because targets (and, with `lock_targets=1`, receptors) are protected, the
candidates sit on the signalling mediators between the NK receptors and the
transcription factors. In an experimental follow-up you would prioritise the
genes that appear in `Best_balanced` and are also differentially expressed
between conditions.

## Alternative: the one-call CLI / Python API

For production runs the same pipeline is available through
`EvoFlow.cli.run_evoflow` (and the `evoflow` console script). It also handles
h5ad cell-type filtering, bulk expression files and automatic output naming:

```python
from EvoFlow.cli import run_evoflow

run_evoflow(
    network=str(network_path),
    annotation=str(annotation_path),
    expression_file=str(expression_path),
    receptors_raw="IL2RB|IL2RG|IL15RA",
    mediators_raw="TBX21,EOMES,STAT1",
    ligands_file="ligands.csv",   # CSV with an 'x' column of ligand genes
    targets="TBX21,EOMES,STAT1",  # alternative to a regulons file
    generations=2000,
)
```

Use the low-level API shown above when you already have pre-processed inputs
or want to keep the `GAOptimizer` object (for example to inspect the active
subgraph or to pickle it for later runs).

## Saving the executed notebook

The committed version of this notebook intentionally contains **no outputs**.
To publish it with results:

1. build/pull the container described in the documentation,
2. run `vignettes/use_case.ipynb` from top to bottom in JupyterLab,
3. save it (`File > Save and Checkpoint`) and commit the `.ipynb` before
   tagging the release.

All figures are already configured at 96 DPI with small figure sizes, and the
only downloaded data are cached outside the repository.